# A CNN from a spec sheet, and its parameter count by formula

Build a small mobile-style CNN exactly as specified, then count its parameters with a formula. The 2026 P8.3 asked for a parameter count with `numel` banned; here everything that could count for you is banned.

```python
class SpecCNN(nn.Module):
    def __init__(self, in_channels, num_classes, widths): ...
    def forward(self, x): ...          # (B, in_channels, H, W) -> (B, num_classes)

def param_count(in_channels, num_classes, widths): ...   # returns an int
```

`widths` is a list of $L \ge 1$ positive integers $w_0, \dots, w_{L-1}$. The model has these attributes, in this order, with exactly these layers (names matter: the grader loads weights by name):

1. `self.stem = nn.Sequential(conv, bn, relu)`: `Conv2d(in_channels, w0, kernel_size=5, stride=2, padding=2, bias=False)`, `BatchNorm2d(w0)`, `ReLU()`.
2. `self.blocks = nn.Sequential(...)` holding $L - 1$ blocks (empty when $L = 1$). Block $i$ ($i = 1, \dots, L-1$) is an `nn.Sequential` of six layers:
   a depthwise `Conv2d(w_{i-1}, w_{i-1}, kernel_size=3, stride=2, padding=1, groups=w_{i-1}, bias=False)`, `BatchNorm2d(w_{i-1})`, `ReLU()`,
   then a pointwise `Conv2d(w_{i-1}, w_i, kernel_size=1, bias=False)`, `BatchNorm2d(w_i)`, `ReLU()`.
3. `self.pool = nn.AdaptiveAvgPool2d(1)`, `self.flatten = nn.Flatten()`.
4. `self.fc = nn.Linear(w_{L-1}, num_classes)` (with bias).

`forward` applies them in that order. `param_count(in_channels, num_classes, widths)` returns the number of learnable parameters of `SpecCNN(in_channels, num_classes, widths)` **computed from the arguments**, without building a model.

**Example.**

```python
model = SpecCNN(3, 10, [16, 32, 64])
model(torch.randn(2, 3, 32, 32)).shape     # torch.Size([2, 10])
param_count(3, 10, [16, 32, 64])           # 5162
```

**Restrictions.** Only `torch` (and `torch.nn`). Banned names anywhere in your file: `numel`, `nelement`, `parameters`, `named_parameters`, `state_dict`, `children`, `modules`.

**Constraints.** $1 \le$ `in_channels` $\le 4$, $1 \le$ `num_classes` $\le 1000$, $1 \le L \le 6$, $1 \le w_i \le 2048$, inputs at least 16 by 16. `param_count` must take under 0.1 s even for the largest widths. CPU.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem2_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem2_submission.py
import torch
import torch.nn as nn


class SpecCNN(nn.Module):
    def __init__(self, in_channels, num_classes, widths):
        super().__init__()
        # Your code

    def forward(self, x):
        # Your code
        return logits


def param_count(in_channels, num_classes, widths):
    # Your code
    return total


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem2_submission as submission
importlib.reload(submission)
# e.g. submission.SpecCNN(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9Wwtz3DaS/itYpWpFekf0zMj2peQod7KjZH3lV1lO7V5JKgYzxMxwxQEZgmNJq9V/v68bAAnOQ35k75yUhgSBfqG70d0A7vbybO9I7K1G44NqON4biL2laiSa7ja/NHlTKGo7ES/fvhWzulwKKUylpsIslGoGQupM5I0Rlawl4KhaTMuVbsTkVszKerkqJMEplDGl9sCL4SGjzfWqUQat4yFes3w2y6erorlFyyEalG5qet47Az6g5zFltipUqoGLvlR1OSnUcpya1WSZG5MDCXrJoiivVZbmy6qsG8JwvteU9XSxd4mvE6m1yrhRr5aqoBEt+YbeCHyW9ttMIxuVZvm04R4KaEEgPU8XeZHVSnf0GY8nLcqyIvwzWRhFAlWmSZt8qcpVk7asN3JuiZxqnbRoU5YkY0DzvC5XFVPCnCQWU/c+LfWnccaITbmqpywfmcmqUdnBeDh+dvBhdPD+e8dsM8bnpl6pe7zP8oKn4W5vIWuNqUoqlvsF//dXVVSQgWgWshFO4GJey4warYRF9BurhhvuWx/9FicX+kKfuO4iN9AO3UBuj1caWvP4B/r58bGDivcqz358bHuDiES8akSmZjmgChKdmK30tMEsG8FzxI3pI0tbI68UntSF7rRBWDGJ39D0WyJO/37y5v3r0zNR5KYhlpQFS8AMVLq81gKDPqy0iD7lJgdV8XNxRuAaUa+AVn1S9S0PAms/y7xY1RhZy9woQa/iOm8WsBHAM3KuxHWdN2CYjEG8UPpIGHkrronca2nwtdTzgdAEVWSrZSUWeZahe66rVWNIenYWtBPptKxu2xfSI5Kve4UyV7cCUHVFrd+Jl2UhJ+Ll+1+NkLUShoyiJllpImXfiDdj8b4uB8KUoiinsiCIuZ4bsVxBKBNFQoWISjLMHJ+LnOQA024WmMqZnEL1EsI0ToY0uVLMVzT/kcVsAKtQB82iVjIjr4EJQy8o8nxR3EJbihn8B1ExLWsSdK1I6I3IZ0QgXtl8HR/OakgoZ+9PT39Kfz55+fHdB3FM2InhCz0tJLDTNESnN1NVkabERxda4F+FT7YXFAoSV1kEXcwGYmnmvg/w6pKErDPXQv+6yY2obwck1VV048fCVwVj/AyRaQatM/Cfa7gSPVXRzcB+Tz4qbco6DoYzWtWsai1ukgzuebqI4mRarfCXpzmKbWfFbIpXjO60rss6gGJZDmDpKpFG1rW8Bd0dH2YhKxXNS7jza6npL/Tz+GIPmg13fLHnKTOQdbOqChUR5+gfJzzU0cIyRR/fiWDFAzG72LsjgPeYcGNxiTsDv6Nu4NfhncRd0P8e6DrKpkVpHqJsIOqmLI5H6uAZliL3+L2nF5Z1DZJbYqVpbsHorCglEUbtjLP/IWBmbvkjlq7t4w52LAOue7yNN/+J2WsRYEJyA/eWNxgNOooiigU8BWlh+O3afQvRkyeVOXtCTWNyPWuBO0Wm+S4KJ0QIw4mL/jhx0Z9Q8VRdQ2IsCCJuKW/oR05MNBcH4jqOY4INRvN/KqGwpokhGd8WU+nLiR2d+CQL+AcRFbKek9tVpLDiDj9HyeH8Pu7Nvfp9JYsv0Mq+UdmuRLTOWNMHMUsURMimqd33iz2ejg7IFnUJFKTr83+uFqFqMP2pkwOI2zCmUKotBJqVo3WKoQxELTPfwYAzvsOnP9U90qgXmnqTkZuUTYSng57MzvlghMGMdEM26KdWiMIBjpOUA7s07ZEzh/ujmXNY932nfQ+1T2lqtKywkjf/v665KLX6SqcMlF4rb0gleW3fizmkJuvt0xNR6DngwCUO6ekz16OOAEaZUtXxR0R6gYI5Aj9CfOvkbQKIe0sINSUEk7/d9ASPaYnkQEy+Xe7yi+Ruu1i7YHzfKnZJYmc4pu8KHCJpkZhoEm/zNRje+Zmtw1s/MXGPPLd9yyYOBtbbQbN1OFceDkMIgpcyXa5gFIhuopkeiEdwp7CMR4+urunJU2JDx5dYAcRMM+YZBajEA2IcRIFzmJfUCBnr+YqSGSwhoirkVCU+6uRVn+wJfvG8sywZczwraQBhvLQ9EZ3R4qGjdYLsZxqSQ5HBsIlpqKL0q0ZaFf0zryI7hnH0FJz9SatczoncIsWBLmaKAvx8llNkSSFzx8t3d7n4ixjdPxfXZX1FUb1k5RU0fYhHW3/phAziOwlTEJ/ryCgKBUHVl0r5eShieNcw5N4SSXOeivXQIPYGByQgjnX325EI3m0ORajeAURNKRgSpKUhljgOfyOnYgoENISyc466KVvhwJ6CA2QXB5NVBk9KzRyDIyzPKYFaVTAn0VznU0UAIaEcK3Q5cxYMmr1EDasQtcwQL9HyTRBsum8S8ZE+5DWawUyxTyhoXbDISMIq66lVM6RoEhASZJczm+uq2nubz6hS1mwfjBilceGI4/dYuEkUj0WYNnQqmZIq1mQN0TjUO0wi0Pxw7CCtuaIJcpqrrkmOB+JqDGx9/8gkD9Yae5x8XhRMJwlizGIYB80sBmRsUUZL7A5xhPFmyBDbUVOWrDdI/rPmKBnP7hHlQ8vUc5d1tUrMygSzY0BipSmfv3OiceM4h0Ri56d4yz9GicwSafAMqjWnANYmmXdunhgQWceDBlrrOaxTZX7CugQHM5mVS0h7JqF+advxguszKyNlXqZcYehVOX6hFlgAKOuKBzk7DSrvkNVNqTKChDknW6Ce/3327i1o1Ww5pslAIdtqddss0HLwSng84gdX4kizvP5R/NDhwEe8wyr/xW3NjxspP4wtSP8xsTdNkU/aJvuDlmTV5EXXXLaP/zBlVySoZLMIh5t8rmU3zNyatfKCf6mxLkzk9KptgRZrKhbYWoOZ5mBTq9bdoeEqbw4KhW5ilDxLRi6mM0dUJ1pSSdCWD5FduMXHO79K5pDy/EJ7HMksL6DR/jW62APZZa0oA3SVFkSiyaNf9ZWmGo4pC6qolFwBAMK8ollLHrlI8a+nH05hOU4WyXv8RmlKZbA0jRM4LhqOqK6SNSi70JBKQp0TaKuqm2gIY0D4QGBiANxW++JakPgOa/Xv8kicPhmOuRL2+m8n/3OWvjh5+/b0J1Bwd7GXuiplmhIzYMzPp31VN2rqnhDn26dpuaxA68Xe/YV+/e7d+/Ttu59OzwAugq4kP5ewVXr4GxRX2ccTc6un7YfXiCFfAoZ9O1PBy08IMru3X5Sm9bmsT2+qOCyxfLS1mM9VWVKJTG8ZkY6tlvA3VNhsLZZTRQ8orEZQZDNNpws1vTKRqac0xY3sr7cfrL1TNIzFiuRYXlnhLM0cgnkOt0gRoK3xsdmYZr+dHQ6GXCmRFqqmzm1ZsbdE2UXo/HJbNNvUCoEdSwlagvwahPbj0LNb5Oc3HIdSUU5txofnjuwjaAuXhx31R+QmDQ93KXKprZ+5Uwn96vL+iJ7RG8nPvSPQVb1pSYC8Ejhm2MlaKZxkdH7pCJ3Itp3XycYvOVwOTVeGgQXtE9hDamulrp1UHCEjbDfPYCoUidU2ULCAAcH2jyijoaA7n1DRH+xcceymioQU2yHgcnkg86VslqsibJlxdQSumnwzhH8ti6uIJmNt6Q7CdOpu9dmmButZRRvKUseEyT3aXMBcSQXxciS5EwXlSyot2NcE2CBIihyjJSJPuCNK5gj4kom1ExEfbV8bg7lIZJY5HPFm525ugn6JqYqcpptwnh+MLrcM7CYvWVUZRd1bxoZJYvGwGH+G11vnBhOKuWI5usI72O8MaosseYQTJJ7/gBTXJQhoa1IIRNeTwDYNWBu6S3jfNJT5/Fqhn3jbWed9TSGYDjK0L4L6NvDIDwHMs91cbe+0DWO3WK3jZMNPZFUpnUW+4mMHXexZl2fd+3/25bUNC6+CL3L9zq1iJ6v5iaE1KLa1lbUBSen6vZHNG4SMGzptfdBXEgfCAoUMQFKQ6EDtdP8+CrBWYlWeWtkgqF/yj5LSU3awUYAnjrd7l7/QigLBeFB3+wOxb4F4iyID3Ac+5L3zfSrF3m+vJ26jn0oWW8g3nvYO2HftDuJEakOynS7cCsGhYLdAcPlZSSSxf+ff+LnNRotreWsO7B4uMhnEhXkLiJTSQSt1cTvwOJG5dEtNZLXA75pyKEVBx60o1Byp+TWFTYJ7JkkSB4sZgjKRlTQpFvAi5/zSzkIULJp/5sUxWIUtuW7xhRr+S0TBWvpn0YsLAxUChq9WHVBghJMPwY9MvEV3ADn+IzOsS4+j3ZLtZhmU95lPrRGlZWXjyX61jbpbI1szvS9hmDJK6MV/CQ+bUtbAbkRkIyeH4Xx4GSr3Fva/RgQd1oeYZ+e2wTK3fhvDNkxiZe8ximYyY1LqWi2U5iS2jR151L+Xf8fZ/QOZ+qzOAaa4TTmSjQACiW5K2VQ/on+n1QHTmSkzrXPOKkQFVYV9z4XbieYOud++poQN5qqon+HN+n6FaUJlFZ+3Jsib8dykzSRSSZq2H5DyuWCTgCPSnPGiPiM8AIGZmlH6qXzMR5lfy4KLSq+pZEIFK/KyXJt0Soe/FI4FUXuviaaCFYZ683ZaR74TJu2Y8A6JCvZHKPq/v2Os99TfC3uJSDxqczF42vUslzJZWc8/nbchomdlZ8/xZZASO0lRAH4s2i6HPlyH3qOdCg5JUcrMRBET8ZjUhmyCSxF7BA9rFlUyIu/skD4RREdM2MFLg6uJx2LTLmzCSE3nl/ROqZ7pXm1hxqqsy6RJE6nhbYkpCOGfd9AuiZwdqWhr6BTBkgmac0vVJa8W9MZWchmGsz6l8/Gs4brjBtrQS3DVImJx0uEQE9nu4SpvtcSXPGcgOi+Ts4bKJ6/eeeHZIk9ifyL3dvbql5PXH94MXJbuusKdpVzc9bvAgS/rH14iWY6HnpZ+dsyHYLpiFaYzy2sF47OTEYHS9TDL06ianFDUnsxXH1+9Of2Qfjg9eT1oqVsLc/hA2vFaKSyh1pRqM7ayQ9VJ3jIJOAqOk1nV6ApzmKbAVa3hW0020TlYjJBQR/RnfSAsxh0RO6fPnDBcWsXfZIltSNUJ1YBSO4w8T/wNghv2axOu7NIrSThFdMZxab2Z5ZE8MHs19rhNWV7xnoK48/NxlAypAuxdl8PyQhrVVohsDaQrib14fTocjjoCvoqNXRS7jSEmk3Zi6IRY6SbKBUEPrEetWW+A3iYo71cIL/SZNIp34iMk4E+Gw+HR5TcbMp987NdxuCk832H7+L1iv6MJXgb2U/z5ye1klZXKBk32sB1WFwJxH+btX8nC/Ots8mLPHvojK2yXjPYcYMu3O0f4edubB8Y332VLFprr5Y77ucKWz/JsF/IM/uigD+Kd26O1hoKGCInKzDp2eiLf/knWxuNI8kYtTcSnZ3R/VWCv6vf9aQNFgoxoFocFL3gJ2oTkkIQwrgXPvByT9tcr+C2rFBSt8CkC7Vlbc7jknnO9UuH8Te0Ca/3hEQM5f3p0Cf/N+8Kg1lWpBbvHiz0H2ncOsNFXqgZT2rMtTXH+1m76UNMwGd5/zZ7Y5sGHb1h1/ujK4zbnNr3yN7k0NwnnneTIUinw3jy7seHBu8E29rA2zs66KPXc78us+2wR5dqeMuNofkAJxEzWwu8MxqETcMh5c2Otpv0HGW5ppvB6y1mVr1pL/l20hOcMeL8i+8I1pIPVKjjBq6HFWbRro3YgDrctby6cvfQJWd06tq9eitoBNlY9ptQx6mLXh6JSdiqTsiyiDcLsNwLV9DSXwDU9cO0In6DsXFL4WPNM+ITHerc0pfyGNsqO/PYApTt2Z7ddLGhPd+sOpj14FL4lmlVJ6917eAPe7x7Yk6gDdzAk2IdWM+RgVEoEYXSmRiOLG9BR7JR3yvicXJ41i/a8yJIK5Dp5Y5cgn1BhSVDuy5n6fUVbKYjU8faSj/X3oVuAyOQH4invROaZOh7DR8ksQ6hGj5NcmmN2unDTAPOC6mNvy3oJWO1w++mDev1rm4YtkwnW5CuzQcqj8/57cD6nJVIOBP4/3EbSaCDsBYZj+TBxMiRqsAMLYpzRw2AmPd5slR4jSBvprJETwRHy8Faco6PL+LIVQ1XSGdllMisknd/nx6mVygldqsg/qZNP8/foBWwji+1n2zeyb68RSMnao2JMgVr0KybLnkalIPda1lm0HIibtRMOREbUkhVZQiM/bZFVpOgmpn8dULrpYb3NF+upw2dWy6hK+J5MZOVYWZP+Yr3vLrSYyNPU3sNA5OQiIHUjlxVFEpcd2VTezWxMa48BITZS7Xlldw9Gajo5R2c8omFQnygA2x3jepoM2UMn7iKRO8sVbz+vyoMHnY26U6uM+v5IOBj2igSA2g3xtvf6qRWGFhxop1JCyo0hX3Scep09z8QOITjiOXfcOkW24r0mfA6TpObjU51ObIhD908Le96pBGfJotO7d/q+u/YFuugrR/1zGIex54jvA3lQcaLTmxDnvGzSK3Vr7ElvfkRf2ty3s5F0l6+iOKHPYMaehcWaMdv6uU2UPHDxp+MOfHjkneqCpIhXLLsrkl07BuOvfBjdDr48PzoMkjsuGCBD7oPoOAlheMBrINbOzXt5d3wJhkRX5BDH8VrVSjta6e6wNDNzP2hpunMP9/Fzq3vdFkuw7VLIWzotU3MC5rWEgmm3StoDU1EYPD+gZmtBYcXHiNPIwqLjWTql3f+oisUjRP1PY9/zOyzoV0pM7KYQHLngg5GWRrPIZw0lqfoADHzK6dBSSIzNkagE1hJlr/GhabaNLsrEaPsbMZVPxZA+0fmiFJlbv1zf7noiuFrnxjJDuQUiQ2KnP4z3Xnchoo2tL8ekA1QkuVFnWpzepoEdbNpN6BvdRthzHmqfu2xzQfcgIER2SgFpN+Q0OmKisXVIHIcs+HbJ3J8XOrb92vNDMMvQTy9A/mg4hLyuw/rBZ1SuO5Jj/efNmqTZ4myHcBHFMy2jaweKA4ePIWvHzu0GId/QoJ0Qe0ejf7Ep7T4M1mavs+CBO5R8t7i/ubuGQWMQvfevh7hSA5Oxz5/2BwLzFe+4IxWg7t8bsRd9GI7vN3iImqKc09XdKDB/yv6QMjV8C3Cbu4nDS1dPuktXT+P1Rc4uLTsX7wfze76vwks2O5Yty9QXnADeehEFGcfWGxb8kfKbteU+wG+XfH8mn4oxjeCbNdtusMSdI/38crtxPWcbdn+Kmq/sbFzYWbtYRgd9yUPsBseFge7Q73PeNaRlIW+6GW/Pfveu2ITxGpc+3JxuBDcRcoERYq/z0TPkBfAXz55Q0nEe0fPhOL60JxwZkA+sto5pmdtQrR0oOrChAu0E3eOMGTAPMzbi1Ov8e8sOwRs9w2NE17kPR5Yxd5XXBkzs2+jQMEfFdBxO2JB9nbcQC0j8D8A/ZGR4ctIb4XF8SOiewQKfDHehm1HxwkbhDyB5winb+Yjk0bHz5PttUBdwMA4inBjS96tN8dkLAaH43MUP0vr+tY/znhy3s0v0Ya2gaSNuR+PvwfxTUPiUKB4Nx09oG+oJQdh56tz94/knVvlnxNDxADDnBMHD6W3B7fBls1Yy1prcZROWxt79/f8CV4kQkg==")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
